# Generate well-level bulk profiles profiles

NOTE: We are normalizing the bulk-profile plates to the negative controls as the "standard".

## Import libraries

In [1]:
import pathlib

import pandas as pd

from pycytominer import aggregate, annotate, normalize, feature_select


## Set paths and variables

In [2]:
# Set the plate to process
plate_id = "CARD-CelIns-CX7_260803130001"

# Directory with QC-labeled profiles
qc_labeled_dir = pathlib.Path("./data/qc_labeled_profiles/").resolve(strict=True)

# Directory to save bulk profiles
output_dir = pathlib.Path("./data/bulk_profiles/")
output_dir.mkdir(parents=True, exist_ok=True)

# Path to the platemap for the validation plate
platemap_path = pathlib.Path(
    "../metadata/platemaps/platemap_validation.csv"
).resolve(strict=True)

# Path to the QC-labeled profile for the validation plate
profile_path = (qc_labeled_dir / f"{plate_id}_qc_labeled.parquet").resolve(
    strict=True
)

# operations to perform for feature selection
feature_select_ops = [
    "variance_threshold",
    "correlation_threshold",
    "blocklist",
    "drop_na_columns",
]


## Process data with pycytominer


In [3]:
print("Performing preprocessing on", plate_id)

# generating all output file paths
output_annotated_file = str(output_dir / f"{plate_id}_bulk_annotated.parquet")
output_normalized_file = str(output_dir / f"{plate_id}_bulk_normalized.parquet")
output_feature_select_file = str(
    output_dir / f"{plate_id}_bulk_feature_selected.parquet"
)
output_spherized_file = str(output_dir / f"{plate_id}_bulk_spherized.parquet")

# loading profiles
profile_df = pd.read_parquet(profile_path)
platemap_df = pd.read_csv(platemap_path)

# Drop all rows in the profiles that failed any Metadata_cqc columns
cqc_columns = [col for col in profile_df.columns if col.startswith("Metadata_cqc")]
if cqc_columns:
    profile_df = profile_df[~profile_df[cqc_columns].any(axis=1)]

# Step 1: Aggregate single-cell data to the well-level using the median
print("Performing aggregation for", plate_id, "...")
aggregated_df = aggregate(
    population_df=profile_df,
    operation="median",
    strata=["Image_Metadata_Plate", "Image_Metadata_Well"],
)

# Step 2: Annotation
print("Performing annotation for", plate_id, "...")
annotate(
    profiles=aggregated_df,
    platemap=platemap_df,
    join_on=["Metadata_well_position", "Image_Metadata_Well"],
    output_type="parquet",
    output_file=output_annotated_file,
)

# Load the annotated parquet file to fix metadata columns names
annotated_df = pd.read_parquet(output_annotated_file)

# Rename columns
annotated_df.rename(columns={"Image_Metadata_Site": "Metadata_Site"}, inplace=True)

# Save annotated profiles back to parquet
annotated_df.to_parquet(output_annotated_file, index=False)

# Step 3: Normalization (mad robustize)
# Normalize using the negative controls as the reference population
print("Performing normalization for", plate_id, "...")
neg_control_query = "Metadata_treatment == 'DMSO' and Metadata_cell_type == 'failing'"
normalize(
    profiles=annotated_df,
    method="mad_robustize",
    samples=neg_control_query,
    output_type="parquet",
    output_file=output_normalized_file,
)

# Step 4: Feature selection
print("Performing feature selection for", plate_id, "...")
feature_select_df = feature_select(
    profiles=output_normalized_file,
    operation=feature_select_ops,
    na_cutoff=0,
    blocklist_file="./blocklist_features.txt",
    corr_threshold=0.95,
    freq_cut=0.05,
    output_type="parquet",
    output_file=output_feature_select_file,
)

# Step 4b: Remove features with too little variation inside the exact control
# population used to fit spherization.
print(
    f"Feature selecting {plate_id} with variance threshold "
    "within negative controls only..."
)
zero_negcon_var_fs_df = feature_select(
    profiles=feature_select_df,
    operation="variance_threshold",
    freq_cut=0.05,
    unique_cut=0.01,
    samples=neg_control_query,
)

# Step 5: Spherize/whiten all profiles using the negative controls as the
# reference population.
print(f"Sphering {plate_id} using negative controls...")
normalize(
    profiles=zero_negcon_var_fs_df,
    method="spherize",
    samples=neg_control_query,
    spherize_center=True,
    spherize_method="ZCA-cor",
    spherize_epsilon=1e-6,
    output_file=output_spherized_file,
    output_type="parquet",
)

print(f"Saved feature-selected profiles to {output_feature_select_file}")
print(f"Saved spherized profiles to {output_spherized_file}")


Performing preprocessing on CARD-CelIns-CX7_260803130001
Performing aggregation for CARD-CelIns-CX7_260803130001 ...
Performing annotation for CARD-CelIns-CX7_260803130001 ...
Performing normalization for CARD-CelIns-CX7_260803130001 ...
Performing feature selection for CARD-CelIns-CX7_260803130001 ...
Feature selecting CARD-CelIns-CX7_260803130001 with variance threshold within negative controls only...
Sphering CARD-CelIns-CX7_260803130001 using negative controls...
Saved feature-selected profiles to data/bulk_profiles/CARD-CelIns-CX7_260803130001_bulk_feature_selected.parquet
Saved spherized profiles to data/bulk_profiles/CARD-CelIns-CX7_260803130001_bulk_spherized.parquet


In [4]:
# Check an example output file
test_df = pd.read_parquet(output_spherized_file)

print(test_df.shape)
print("Plate:", test_df.Metadata_Plate.unique())
print(
    "Metadata columns:", [col for col in test_df.columns if col.startswith("Metadata_")]
)
test_df.head(2)


(60, 1032)
Plate: ['CARD-CelIns-CX7_260803130001']
Metadata columns: ['Metadata_well_row', 'Metadata_well_column', 'Metadata_heart_number', 'Metadata_cell_type', 'Metadata_heart_failure_type', 'Metadata_treatment', 'Metadata_Plate', 'Metadata_Well']


,Metadata_well_row,Metadata_well_column,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Plate,Metadata_Well,Cytoplasm_AreaShape_Area,Cytoplasm_AreaShape_BoundingBoxMaximum_X,...,Nuclei_Texture_InfoMeas2_Golgi_3_01_256,Nuclei_Texture_InfoMeas2_Golgi_3_03_256,Nuclei_Texture_InfoMeas2_Mito_3_01_256,Nuclei_Texture_InverseDifferenceMoment_DNA_3_00_256,Nuclei_Texture_InverseDifferenceMoment_DNA_3_01_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_00_256,Nuclei_Texture_InverseDifferenceMoment_Mito_3_00_256,Nuclei_Texture_SumEntropy_ER_3_02_256,Nuclei_Texture_SumVariance_Actin_3_01_256,Nuclei_Texture_Variance_DNA_3_01_256
0,B,2,7,nonfailing,nonfailing,DMSO,CARD-CelIns-CX7_260803130001,B02,-0.051396,-0.019104,...,0.032456,0.025740,-0.016164,-0.054992,-0.108118,-0.078653,-0.011059,0.037030,0.986861,0.044727
1,B,3,25,failing,dilated_cardiomyopathy,J0337,CARD-CelIns-CX7_260803130001,B03,-0.032809,0.006128,...,0.084798,0.059421,0.038710,-0.068882,-0.089541,-0.050963,-0.052787,0.022604,2.195859,0.026489


In [5]:
# Check an example output file
test_df = pd.read_parquet(output_feature_select_file)

print(test_df.shape)
print("Plate:", test_df.Metadata_Plate.unique())
print(
    "Metadata columns:", [col for col in test_df.columns if col.startswith("Metadata_")]
)
test_df.head(2)


(60, 1033)
Plate: ['CARD-CelIns-CX7_260803130001']
Metadata columns: ['Metadata_well_row', 'Metadata_well_column', 'Metadata_heart_number', 'Metadata_cell_type', 'Metadata_heart_failure_type', 'Metadata_treatment', 'Metadata_Plate', 'Metadata_Well']


,Metadata_well_row,Metadata_well_column,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Plate,Metadata_Well,Cytoplasm_AreaShape_Area,Cytoplasm_AreaShape_BoundingBoxMaximum_X,...,Nuclei_Texture_InfoMeas2_Golgi_3_01_256,Nuclei_Texture_InfoMeas2_Golgi_3_03_256,Nuclei_Texture_InfoMeas2_Mito_3_01_256,Nuclei_Texture_InverseDifferenceMoment_DNA_3_00_256,Nuclei_Texture_InverseDifferenceMoment_DNA_3_01_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_00_256,Nuclei_Texture_InverseDifferenceMoment_Mito_3_00_256,Nuclei_Texture_SumEntropy_ER_3_02_256,Nuclei_Texture_SumVariance_Actin_3_01_256,Nuclei_Texture_Variance_DNA_3_01_256
0,B,2,7,nonfailing,nonfailing,DMSO,CARD-CelIns-CX7_260803130001,B02,-1.124398,-1.124151,...,2.45020,1.441764,-0.296467,-3.109958,-8.562595,-2.709610,-0.916670,1.069859,22.689669,4.051827
1,B,3,25,failing,dilated_cardiomyopathy,J0337,CARD-CelIns-CX7_260803130001,B03,-0.809374,-0.020439,...,5.26951,2.806924,0.935465,-3.518994,-7.150726,-1.668974,-3.111689,0.735788,50.282803,2.952129
